# Annexe B — Cahier de code, Chapitre 8
## Géométrie de la caméra

Ce notebook accompagne le chapitre 8 de *Fondamentaux de la Vision par
Ordinateur*. Pour **chaque sous-chapitre**, une cellule de code Python montre la
**syntaxe et l'usage** de la notion — au plus simple, sans fonctions ni gestion
d'erreurs. Le but n'est pas de produire du code de production, mais de relier la
formule du livre à son équivalent Python.

> **Pré-requis** : `pip install numpy scipy scikit-image scikit-learn opencv-python matplotlib`

Exécutez les cellules dans l'ordre : la première prépare les données d'entrée.

## Préparation des données

In [ ]:
# Outils : numpy pour l'algèbre, OpenCV pour les routines caméra
import numpy as np
import cv2

## 8.1 — Coordonnées homogènes

In [ ]:
# ajouter un 1 → la projection devient un produit matriciel
p = np.array([3.0, 4.0])
p_h = np.append(p, 1)            # [3, 4, 1]
# retour en cartésien : diviser par la dernière coordonnée
q_h = np.array([6.0, 8.0, 2.0])
q = q_h[:2] / q_h[2]            # [3, 4]

## 8.2 — Le modèle sténopé

In [ ]:
# x_pixel = K · [R | t] · X_monde
K = np.array([[800, 0, 320], [0, 800, 240], [0, 0, 1.0]])
R = np.eye(3); t = np.array([[0.0], [0.0], [5.0]])
X = np.array([[1.0], [1.0], [0.0], [1.0]])     # point monde homogène
x = K @ np.hstack([R, t]) @ X
x = x[:2] / x[2]
print(x.ravel())

## 8.3 — Calibration

In [ ]:
# à partir de plusieurs vues d'un damier (object_points / image_points)
# ret, K, dist, rvecs, tvecs = cv2.calibrateCamera(
#     object_points, image_points, image_size, None, None)
# image_points provient de cv2.findChessboardCorners sur chaque photo
print("voir cv2.findChessboardCorners + cv2.calibrateCamera")

## 8.4 — L'homographie

In [ ]:
# transformation perspective entre deux plans (4 correspondances suffisent)
src = np.array([[0, 0], [1, 0], [1, 1], [0, 1]], dtype=np.float32)
dst = np.array([[0, 0], [2, 0], [2, 3], [0, 1]], dtype=np.float32)
H, _ = cv2.findHomography(src, dst)
print(H)

## 8.5 — La géométrie épipolaire

In [ ]:
# matrice fondamentale reliant deux vues d'une même scène
pts1 = np.random.rand(20, 2) * 100
pts2 = pts1 + np.array([5.0, 0.0])           # décalage horizontal factice
F, mask = cv2.findFundamentalMat(pts1, pts2, cv2.FM_8POINT)
print(F)

## 8.6 — Stéréovision

In [ ]:
# carte de disparité → profondeur, à partir d'une paire rectifiée
left  = cv2.cvtColor(cv2.imread("left.png"),  cv2.COLOR_BGR2GRAY)  if False else np.zeros((200, 200), np.uint8)
right = cv2.cvtColor(cv2.imread("right.png"), cv2.COLOR_BGR2GRAY) if False else np.zeros((200, 200), np.uint8)
stereo = cv2.StereoBM_create(numDisparities=64, blockSize=15)
disparite = stereo.compute(left, right)

## 8.7 — Distorsion

In [ ]:
# corriger la distorsion de l'objectif avec K et les coefficients dist
K = np.array([[800, 0, 320], [0, 800, 240], [0, 0, 1.0]])
dist = np.array([-0.2, 0.05, 0, 0, 0])       # k1, k2, p1, p2, k3
img = np.zeros((480, 640, 3), np.uint8)
corrige = cv2.undistort(img, K, dist)